### 1. Análise e Tratamento de Dados (Parte 1: Limpeza Exploratória)
Nesta primeira fase do pré-processamento, realizamos a limpeza estrutural da base de dados, lidando com a integridade das linhas e colunas antes da criação de novas variáveis.
* **Duplicatas e Valores Faltantes:** Removemos registros idênticos para evitar viés. Os valores nulos na contagem de avaliações do iFood foram imputados com `0` (assumindo que a ausência indica falta de avaliações). As variáveis booleanas serão extraídas na etapa de engenharia.

In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

print("1. Iniciando Limpeza Exploratória...")

# Carregamento e remoção de duplicatas
df = pd.read_csv('credito_aplicacao_clientes_final.csv')
df = df.drop_duplicates()

# Tratamento básico de nulos
df['ifood_contagem_avaliacoes'] = df['ifood_contagem_avaliacoes'].fillna(0)

print("Limpeza inicial concluída.")

1. Iniciando Limpeza Exploratória...
Limpeza inicial concluída.


### 2. Feature Engineering (Seleção e Extração)
Antes de finalizar o pré-processamento matemático, precisamos transformar os dados em formato de texto e estruturar novas variáveis preditivas.
* **Extração de Booleanos:** Identificação de presença digital (`tem_ifood`, `tem_website_google`).
* **Extração Numérica via Regex:** Conversão dos intervalos textuais de `idade_cnpj` e `capital_social` (ex: "(150-250]") para pontos médios contínuos.
* **Seleção e Quebra de CNAE:** Desmembramento do código da Receita Federal em `divisão` e `grupo`.
* **Engenharia de Credores:** Transformação da lista de strings da Serasa em uma contagem absoluta (`quantidade_credores`) e na extração de variáveis indicadoras (One-Hot) para cada setor de dívida.

In [3]:
print("2. Iniciando Feature Engineering...")

# Variáveis Booleanas
df['tem_ifood'] = df['ifood_contagem_avaliacoes'].notnull().astype(int)
df['tem_website_google'] = df['google_maps_tem_website'].fillna(0).astype(int)

# Transformação de Intervalos
def extrair_ponto_medio_robusto(intervalo):
    if pd.isna(intervalo): return np.nan
    numeros = re.findall(r'\d+', str(intervalo))
    if len(numeros) >= 2:
        return (float(numeros[0]) + float(numeros[1])) / 2
    return np.nan

df['idade_cnpj_continua'] = df['idade_cnpj'].apply(extrair_ponto_medio_robusto)
df['capital_social_continuo'] = df['capital_social'].apply(extrair_ponto_medio_robusto)
df = df.drop(['idade_cnpj', 'capital_social'], axis=1)

# Quebra do CNAE
df['cnae_divisao'] = df['cnae_codigo'].str.split('.').str[0]
df['cnae_grupo'] = df['cnae_codigo'].str.extract(r'(\d{2}\.\d{2})')

# Engenharia da Serasa
df['serasa_credores'] = df['serasa_credores'].fillna('')
df['quantidade_credores'] = df['serasa_credores'].apply(lambda x: len(x.split(',')) if x != '' else 0)
credores_dummies = df['serasa_credores'].str.get_dummies(sep=', ')
credores_dummies = credores_dummies.add_prefix('deve_para_')
df = pd.concat([df, credores_dummies], axis=1)
df = df.drop('serasa_credores', axis=1)

print("Extração de features concluída.")

2. Iniciando Feature Engineering...
Extração de features concluída.


### 1. Análise e Tratamento de Dados (Parte 2: Pré-processamento e Separação)
Com todas as variáveis criadas, retornamos às exigências finais de tratamento de dados estabelecidas no escopo para deixar a base pronta para os algoritmos de Machine Learning.
* **Tratamento de Outliers:** Aplicação do Intervalo Interquartil (IQR) nas variáveis contínuas geradas na etapa anterior para limitar extremos.
* **Conversão de Categóricas:** Utilização do método `get_dummies` (One-Hot Encoding) para converter features de texto em numéricas.
* **Separação de Conjuntos e Desbalanceamento:** Divisão da base em Treino (60%), Validação (20%) e Teste (20%), utilizando a estratificação (`stratify=y`) para garantir que o desbalanceamento da inadimplência seja o mesmo em todos os conjuntos.
* **Normalização:** Imputação final das medianas nos dados numéricos restantes e padronização da escala com `StandardScaler` (ajustado apenas no treino para evitar vazamento de dados).

In [4]:
print("3. Iniciando Pré-processamento Final e Separação...")

# Tratamento de Outliers (IQR)
colunas_numericas = ['capital_social_continuo', 'idade_cnpj_continua']
for col in colunas_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    df[col] = np.where(df[col] < limite_inferior, limite_inferior, df[col])
    df[col] = np.where(df[col] > limite_superior, limite_superior, df[col])

# Conversão Categórica para Numérica
colunas_categoricas = ['uf', 'natureza_juridica', 'cnae_divisao', 'cnae_grupo', 'fonte_cliente', 'segmento_cliente']
col_cat_existentes = [col for col in colunas_categoricas if col in df.columns]
df = pd.get_dummies(df, columns=col_cat_existentes, drop_first=True, dtype=int)

# Isolamento do alvo (y) e matriz de features (X)
df_numerico = df.select_dtypes(include=[np.number])
X = df_numerico.drop(['inadimplente', 'id_cliente'], axis=1, errors='ignore')
y = df_numerico['inadimplente']

# Separação com tratamento de desbalanceamento via stratify
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Imputação de nulos remanescentes e Normalização (Apenas após a divisão)
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Pré-processamento finalizado! Dados prontos para a Etapa de Modelagem.")
print(f"Dimensões do Treino: {X_train_scaled.shape}")

3. Iniciando Pré-processamento Final e Separação...
Pré-processamento finalizado! Dados prontos para a Etapa de Modelagem.
Dimensões do Treino: (1800, 210)
